# Asignación de atributos de TransCAD a vialidades no principales de OSM

Esta notebook asigna los atributos de las vialidades no principales provenientes de TransCAD hacia la geometría de de OpenStreetMap.

El procedimiento combina matching geométrico, propagación por nombre, asignación por zona de dos escalas e asignación global por tipo de vialidad.

In [1]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

### Lectura y preparación de las redes

Cargamos las redes viales de TransCAD y OSM. La red OSM se reproyecta al sistema métrico de TransCAD para realizar todas las operaciones en metros.

In [ ]:
ruta_datos = Path("/Users/.../datos_asignacion_atributos")

datos_transcad = gpd.read_file(ruta_datos / "AMG_RedVial_2023_con_limites_velocidad.shp")
datos_osm = gpd.read_file(ruta_datos / "edges_visum.shp")
datos_osm_metricos = datos_osm.to_crs(datos_transcad.crs)

columnas_atributos = ["CAPACIDAD", "CARRILES", "Velocidad_", "Limite_vel"]

for col in columnas_atributos:
    datos_transcad[col] = pd.to_numeric(datos_transcad[col], errors="coerce")

### Selección de vialidades no principales

En TransCAD seleccionamos los *links* con límite de velocidad menor a 40 km/h (véase la notebook 06).

En OSM conservamos únicamente los *links* cuyos valores de `highway` pertenecen completamente al conjunto formado por `unclassified`, `residential`, `living_street` y `service`.

In [3]:
# Seleccionamos los links TransCAD que no pertenecen a la red principal
transcad_no_principal = datos_transcad[datos_transcad["Limite_vel"] < 40].copy()

# Tipos OSM no principales que queremos caracterizar
tipos_no_principales_objetivo = {"unclassified", "residential", "living_street", "service"}

# Normalizamos highway cuando contiene un valor simple o una colección
def obtener_tipos_highway(valor):
    if valor is None:
        return []

    if isinstance(valor, (list, tuple, np.ndarray)):
        return [str(tipo) for tipo in valor]

    return [str(valor)]


# Un link pertenece al conjunto objetivo si todos sus valores highway son no principales
def es_no_principal_objetivo(valor):
    tipos = obtener_tipos_highway(valor)

    if len(tipos) == 0:
        return False

    return all(tipo in tipos_no_principales_objetivo for tipo in tipos)


osm_no_principal = datos_osm_metricos[datos_osm_metricos["highway"].apply(es_no_principal_objetivo)].copy()

print(f"Links TransCAD no principales: {len(transcad_no_principal):,}")
print(f"Links OSM no principales objetivo: {len(osm_no_principal):,}")

Links TransCAD no principales: 131,494
Links OSM no principales objetivo: 401,264


### Orientación geométrica

Calculamos la orientación principal de cada segmento utilizando sus extremos. El ángulo se expresa entre 0° y 180° para considerar equivalentes ambas direcciones de un mismo eje vial.

In [4]:
# Calculamos la orientación geométrica de cada segmento
def calcular_orientacion(geom):
    if geom is None or geom.is_empty:
        return np.nan

    coords = list(geom.coords)

    if len(coords) < 2:
        return np.nan

    x1, y1 = coords[0]
    x2, y2 = coords[-1]
    dx = x2 - x1
    dy = y2 - y1
    angulo = np.degrees(np.arctan2(dy, dx))

    return angulo % 180


transcad_no_principal["orientacion"] = transcad_no_principal.geometry.apply(calcular_orientacion)
osm_no_principal["orientacion"] = osm_no_principal.geometry.apply(calcular_orientacion)

print(f"Orientaciones válidas TransCAD: {transcad_no_principal['orientacion'].notna().sum():,} / {len(transcad_no_principal):,}")
print(f"Orientaciones válidas OSM: {osm_no_principal['orientacion'].notna().sum():,} / {len(osm_no_principal):,}")

Orientaciones válidas TransCAD: 131,494 / 131,494
Orientaciones válidas OSM: 401,264 / 401,264


### Generación de candidatos espaciales

Creamos un buffer de 25 metros alrededor de cada link OSM no principal. Los links TransCAD que intersectan dicho buffer se consideran candidatos para transferir sus atributos.

In [5]:
buffer_m = 25
angulo_maximo = 35

# Creamos buffers alrededor de los links OSM no principales
osm_no_principal_buffer = osm_no_principal[["geometry", "orientacion"]].copy()
osm_no_principal_buffer["_idx_osm"] = osm_no_principal_buffer.index
osm_no_principal_buffer["geometry"] = osm_no_principal_buffer.geometry.buffer(buffer_m)

# Preparamos los links TransCAD utilizados en el matching
transcad_no_principal_match = transcad_no_principal[["geometry", "orientacion", "CAPACIDAD", "CARRILES", "Velocidad_", "Limite_vel", "TIPO"]].copy()
transcad_no_principal_match["_idx_transcad"] = transcad_no_principal_match.index

# Generamos candidatos mediante intersección espacial
candidatos_no_principal = gpd.sjoin(osm_no_principal_buffer, transcad_no_principal_match, how="inner", predicate="intersects", lsuffix="osm", rsuffix="transcad")

print(f"Candidatos espaciales brutos: {len(candidatos_no_principal):,}")
print(f"Links OSM únicos con al menos un candidato: {candidatos_no_principal['_idx_osm'].nunique():,}")

Candidatos espaciales brutos: 1,377,718
Links OSM únicos con al menos un candidato: 314,454


### Filtro angular y selección del mejor candidato

Eliminamos candidatos cuya diferencia angular sea mayor o igual a 35°. Después calculamos la distancia real entre las geometrías originales.

Para cada *link* OSM seleccionamos el candidato con menor distancia y utilizamos la diferencia angular como criterio de desempate.

In [6]:
# Calculamos la diferencia angular mínima entre dos segmentos
def diferencia_angular(a, b):
    diferencia = abs(a - b)
    return min(diferencia, 180 - diferencia)


# Aplicamos el filtro angular
candidatos_no_principal["dif_angular"] = candidatos_no_principal.apply(lambda fila: diferencia_angular(fila["orientacion_osm"], fila["orientacion_transcad"]), axis=1)
candidatos_no_principal_filtrados = candidatos_no_principal[candidatos_no_principal["dif_angular"] < angulo_maximo].copy()

# Recuperamos las geometrías originales
geom_osm_no_principal = osm_no_principal.geometry
geom_transcad_no_principal = transcad_no_principal.geometry

# Calculamos la distancia real entre cada par candidato
candidatos_no_principal_filtrados["distancia_m"] = candidatos_no_principal_filtrados.apply(lambda fila: geom_osm_no_principal.loc[fila["_idx_osm"]].distance(geom_transcad_no_principal.loc[fila["_idx_transcad"]]), axis=1)

# Seleccionamos un único candidato TransCAD para cada link OSM
mejor_match_no_principal = candidatos_no_principal_filtrados.sort_values(["_idx_osm", "distancia_m", "dif_angular"]).drop_duplicates(subset="_idx_osm", keep="first").copy()

print(f"Candidatos después del filtro angular: {len(candidatos_no_principal_filtrados):,}")
print(f"Links OSM con match directo: {len(mejor_match_no_principal):,}")
print(f"Mediana de distancia: {mejor_match_no_principal['distancia_m'].median():.2f} m")
print(f"Mediana de diferencia angular: {mejor_match_no_principal['dif_angular'].median():.2f}°")

Candidatos después del filtro angular: 617,644
Links OSM con match directo: 271,240
Mediana de distancia: 0.50 m
Mediana de diferencia angular: 1.06°


### Transferencia directa de atributos

Creamos las columnas finales y transferimos los cuatro atributos del mejor segmento TransCAD asociado con cada *link* OSM.

In [7]:
# Creamos las columnas finales de la red OSM no principal
osm_no_principal["cap_final"] = np.nan
osm_no_principal["carriles_final"] = np.nan
osm_no_principal["velprom_final"] = np.nan
osm_no_principal["vel_final"] = np.nan

columnas_finales = ["cap_final", "carriles_final", "velprom_final", "vel_final"]

# Transferimos los atributos desde TransCAD
for _, fila in mejor_match_no_principal.iterrows():
    idx_osm = fila["_idx_osm"]
    idx_transcad = fila["_idx_transcad"]
    osm_no_principal.loc[idx_osm, columnas_finales] = [transcad_no_principal.loc[idx_transcad, "CAPACIDAD"], transcad_no_principal.loc[idx_transcad, "CARRILES"], transcad_no_principal.loc[idx_transcad, "Velocidad_"], transcad_no_principal.loc[idx_transcad, "Limite_vel"]]

print("Atributos asignados mediante match directo:")
print(osm_no_principal[columnas_finales].notna().sum())

Atributos asignados mediante match directo:
cap_final         271240
carriles_final    271240
velprom_final     271240
vel_final         271240
dtype: int64


### Propagación por nombre y tipo de vialidad

Cuando un *link* no tiene match directo, buscamos otros segmentos con el mismo nombre y la misma categoría `highway`. Estos valores faltantes se completan mediante la mediana de los segmentos ya caracterizados dentro de ese mismo grupo.

In [8]:
# Creamos una clasificación normalizada de highway
osm_no_principal["highway_tipo"] = osm_no_principal["highway"].apply(obtener_tipos_highway).apply(lambda tipos: tipos[0] if len(tipos) == 1 else " + ".join(sorted(tipos)))

# Calculamos medianas por nombre y tipo usando links con match directo
medianas_por_nombre = osm_no_principal[osm_no_principal["cap_final"].notna() & osm_no_principal["name"].notna()].groupby(["name", "highway_tipo"])[columnas_finales].median()

# Propagamos atributos a los links faltantes de la misma vialidad y tipo OSM
for idx in osm_no_principal[osm_no_principal["cap_final"].isna() & osm_no_principal["name"].notna()].index:
    clave = (osm_no_principal.at[idx, "name"], osm_no_principal.at[idx, "highway_tipo"])

    if clave in medianas_por_nombre.index:
        osm_no_principal.loc[idx, columnas_finales] = medianas_por_nombre.loc[clave].values

print(f"Links con atributos después de propagar por nombre: {osm_no_principal['cap_final'].notna().sum():,}")
print(f"Links faltantes: {osm_no_principal['cap_final'].isna().sum():,}")

Links con atributos después de propagar por nombre: 284,077
Links faltantes: 117,187


### Asginación mediante zonas de 1 km $\times$ 1 km

Dividimos el territorio mediante una cuadrícula regular de 1 km $\times$ 1 km. Cada *link* se asigna a una zona utilizando un punto representativo de su geometría.

Para cada combinación de zona y tipo de vialidad calculamos las medianas locales. Una mediana por zona solo se utiliza **cuando existen al menos 10** *links* de referencia del mismo tipo dentro de la zona.

In [9]:
tamano_zona_m = 1000

# Obtenemos un punto representativo para cada link OSM
puntos_representativos = osm_no_principal.geometry.representative_point()

# Asignamos cada link a una zona regular de 1 km x 1 km
osm_no_principal["zona_x"] = (puntos_representativos.x // tamano_zona_m).astype(int)
osm_no_principal["zona_y"] = (puntos_representativos.y // tamano_zona_m).astype(int)
osm_no_principal["zona_id"] = osm_no_principal["zona_x"].astype(str) + "_" + osm_no_principal["zona_y"].astype(str)

# Calculamos medianas zonales y conservamos únicamente grupos con al menos 10 referencias
medianas_zonales_validas = osm_no_principal[osm_no_principal["cap_final"].notna()].groupby(["zona_id", "highway_tipo"]).agg(cap_final=("cap_final", "median"), carriles_final=("carriles_final", "median"), velprom_final=("velprom_final", "median"), vel_final=("vel_final", "median"), n_referencias=("cap_final", "count"))
medianas_zonales_validas = medianas_zonales_validas[medianas_zonales_validas["n_referencias"] >= 10]

# Asignamos los faltantes mediante la mediana de su zona y tipo de vialidad
for idx in osm_no_principal[osm_no_principal["cap_final"].isna()].index:
    clave = (osm_no_principal.at[idx, "zona_id"], osm_no_principal.at[idx, "highway_tipo"])

    if clave in medianas_zonales_validas.index:
        osm_no_principal.loc[idx, columnas_finales] = medianas_zonales_validas.loc[clave, columnas_finales].values

print(f"Links con atributos después de las zonas de 1 km: {osm_no_principal['cap_final'].notna().sum():,}")
print(f"Links faltantes: {osm_no_principal['cap_final'].isna().sum():,}")

Links con atributos después de las zonas de 1 km: 366,928
Links faltantes: 34,336


### Asignación mediante zonas de 2 km $\times$ 2 km

Los *links* que no encontraron suficiente soporte en la primera cuadrícula se evalúan nuevamente utilizando zonas de 2 km $\times$ 2 km.

Aquí mantenemos el mismo requisito de mínimo 10 referencias del mismo tipo de vialidad dentro de cada zona.

In [10]:
tamano_zona_2km_m = 2000

# Asignamos cada link a una zona regular de 2 km x 2 km
osm_no_principal["zona2_x"] = (puntos_representativos.x // tamano_zona_2km_m).astype(int)
osm_no_principal["zona2_y"] = (puntos_representativos.y // tamano_zona_2km_m).astype(int)
osm_no_principal["zona2_id"] = osm_no_principal["zona2_x"].astype(str) + "_" + osm_no_principal["zona2_y"].astype(str)

# Calculamos medianas zonales y conservamos grupos con al menos 10 referencias
medianas_zonales_2km_validas = osm_no_principal[osm_no_principal["cap_final"].notna()].groupby(["zona2_id", "highway_tipo"]).agg(cap_final=("cap_final", "median"), carriles_final=("carriles_final", "median"), velprom_final=("velprom_final", "median"), vel_final=("vel_final", "median"), n_referencias=("cap_final", "count"))
medianas_zonales_2km_validas = medianas_zonales_2km_validas[medianas_zonales_2km_validas["n_referencias"] >= 10]

# Asignamos los faltantes mediante la mediana de su zona y tipo de vialidad
for idx in osm_no_principal[osm_no_principal["cap_final"].isna()].index:
    clave = (osm_no_principal.at[idx, "zona2_id"], osm_no_principal.at[idx, "highway_tipo"])

    if clave in medianas_zonales_2km_validas.index:
        osm_no_principal.loc[idx, columnas_finales] = medianas_zonales_2km_validas.loc[clave, columnas_finales].values

print(f"Links con atributos después de las zonas de 2 km: {osm_no_principal['cap_final'].notna().sum():,}")
print(f"Links faltantes: {osm_no_principal['cap_final'].isna().sum():,}")

Links con atributos después de las zonas de 2 km: 384,111
Links faltantes: 17,153


### Asignación global por tipo de vialidad

Los *links* que no pudieron caracterizarse mediante zonas de 1 km o 2 km se completan utilizando la mediana global de su categoría `highway`. Con esto garantizamos que cada tipo de vialidad utilice únicamente referencias correspondientes a su misma clasificación OSM.

In [11]:
# Calculamos medianas globales por tipo de vialidad
medianas_globales_por_tipo = osm_no_principal[osm_no_principal["cap_final"].notna()].groupby("highway_tipo").agg(cap_final=("cap_final", "median"), carriles_final=("carriles_final", "median"), velprom_final=("velprom_final", "median"), vel_final=("vel_final", "median"), n_referencias=("cap_final", "count"))

# Asignamos los últimos faltantes mediante la mediana global de su tipo
for idx in osm_no_principal[osm_no_principal["cap_final"].isna()].index:
    tipo = osm_no_principal.at[idx, "highway_tipo"]

    if tipo in medianas_globales_por_tipo.index:
        osm_no_principal.loc[idx, columnas_finales] = medianas_globales_por_tipo.loc[tipo, columnas_finales].values

display(medianas_globales_por_tipo)

,cap_final,carriles_final,velprom_final,vel_final,n_referencias
highway_tipo,,,,,
living_street,2000.0,2.0,18.500000,30.0,48343
residential,2000.0,2.0,18.500002,30.0,263613
service,2000.0,2.0,18.500013,30.0,62663
unclassified,2000.0,2.0,18.500010,30.0,9492


### Preparación de la capa final

Eliminamos las columnas auxiliares utilizadas durante el *matching* y la asignación por zona. Después reproyectamos el resultado al sistema de coordenadas original de OSM.

**Nota importante:** La capa queda preparada para una exportación posterior, pero en esta notebook no se genera ningún shapefile.

In [12]:
# Creamos la capa final de vialidades no principales
osm_no_principal_final = osm_no_principal.copy()

# Eliminamos las columnas auxiliares
columnas_auxiliares = ["orientacion", "zona_x", "zona_y", "zona_id", "zona2_x", "zona2_y", "zona2_id"]
osm_no_principal_final = osm_no_principal_final.drop(columns=[col for col in columnas_auxiliares if col in osm_no_principal_final.columns])

# Reproyectamos al CRS original de OSM
osm_no_principal_final = osm_no_principal_final.to_crs(datos_osm.crs)

# Seleccionamos las columnas que formarían parte del archivo de salida
columnas_salida = ["u", "v", "key", "osmid", "highway", "name", "oneway", "length", "lanes", "maxspeed", "highway_tipo", "cap_final", "carriles_final", "velprom_final", "vel_final", "geometry"]
columnas_salida = [col for col in columnas_salida if col in osm_no_principal_final.columns]

osm_no_principal_final = osm_no_principal_final[columnas_salida].copy()

### Resultados

Mostramos la cobertura final y las medianas de los atributos asignados para cada tipo de vialidad no principal.

In [13]:
# Reportamos la cobertura final
print(f"Links no principales totales: {len(osm_no_principal_final):,}")
print(f"Links con atributos: {osm_no_principal_final['cap_final'].notna().sum():,}")
print(f"Links sin atributos: {osm_no_principal_final['cap_final'].isna().sum():,}")
print(f"Duplicados en ['u', 'v', 'key']: {osm_no_principal_final.duplicated(['u', 'v', 'key']).sum():,}")

Links no principales totales: 401,264
Links con atributos: 401,264
Links sin atributos: 0
Duplicados en ['u', 'v', 'key']: 0


In [14]:
# Distribución final por tipo de vialidad
resumen_por_tipo = osm_no_principal_final.groupby("highway_tipo").agg(links=("highway_tipo", "size"), links_con_atributos=("cap_final", "count"), capacidad_mediana=("cap_final", "median"), carriles_mediana=("carriles_final", "median"), velocidad_promedio_mediana=("velprom_final", "median"), limite_velocidad_mediana=("vel_final", "median")).sort_values("links", ascending=False)
display(resumen_por_tipo)

,links,links_con_atributos,capacidad_mediana,carriles_mediana,velocidad_promedio_mediana,limite_velocidad_mediana
highway_tipo,,,,,,
residential,268213,268213,2000.0,2.0,18.500002,30.0
service,70245,70245,2000.0,2.0,18.500013,30.0
living_street,48714,48714,2000.0,2.0,18.500000,30.0
unclassified,14092,14092,2000.0,2.0,18.500010,30.0


In [15]:
# Muestra de la capa final preparada
display(osm_no_principal_final.head())

,u,v,key,osmid,highway,name,oneway,length,lanes,maxspeed,highway_tipo,cap_final,carriles_final,velprom_final,vel_final,geometry
6,296347106,5000459113,0,1020599956,service,None,False,15.549603,None,None,service,2000.0,2.0,18.499995,30.0,"LINESTRING (-103.2612 20.47119, -103.26134 20...."
17,325720651,8660442291,0,29555594,unclassified,Reforma,False,33.249600,None,None,unclassified,2000.0,2.0,15.999996,30.0,"LINESTRING (-103.12068 20.57727, -103.12099 20..."
18,325720651,2131769071,0,483591686,unclassified,Reforma,False,22.293186,None,None,unclassified,2000.0,2.0,15.999996,30.0,"LINESTRING (-103.12068 20.57727, -103.12048 20..."
19,325720651,2131777748,0,203194954,residential,Nicolás Bravo,False,271.974304,None,None,residential,2000.0,2.0,18.499903,30.0,"LINESTRING (-103.12068 20.57727, -103.12026 20..."
20,325720652,4763387231,0,29555594,unclassified,Reforma,False,19.868216,None,None,unclassified,2000.0,2.0,15.999996,30.0,"LINESTRING (-103.12119 20.57735, -103.1213 20...."
